In [1]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

True

In [ ]:
import importlib
import eval_utils

importlib.reload(eval_utils)

In [2]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm("hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")

/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.40it/s]
Device set to use cuda:0
/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
100%|██████████| 8/8 [00:00<00:00, 9502.81it/s]


In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, 
                                               chunk_overlap=32,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunks = text_splitter.split_documents(docs)

Initialize Hybrid Retriever

In [ ]:
import time
from typing import List, Set
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.retrievers import BaseRetriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.runnables import RunnableLambda
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document

# set hybrid retriever as base retriever
vector_storage = FAISS.from_documents(chunks, embedding)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

faiss_retriever = vector_storage.as_retriever(
                                search_type="mmr",
                                search_kwargs={"k": 10, "lambda_mult": 0.7}
                                )

class HybridRetriever(BaseRetriever):
    
    base_retriever: BaseRetriever

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        
        docs = self.base_retriever.get_relevant_documents(query)

        # deduplication
        seen: Set[str] = set()
        unique_docs: List[Document] = []
        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                unique_docs.append(d)

        return unique_docs
    
ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever], weights=[0.4, 0.6]
    )

hybrid_retriever = HybridRetriever(base_retriever=ensemble_retriever)

Initialize Multi-query retriever

In [5]:
mq_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        """ You are a helpful research assistant. 
        Given the user's question below, generate **exactly three** alternative search queries. 

        Each query should capture a different aspect or phrasing of the same information need.
        Avoid redundancy, keep them short and clear.
        User question: {question}
        Provide the 3 reformulated search queries, each on a new line:
        """
    ),
)

# Create MultiQueryRetriever，prompt and LLM
mq_retriever_raw = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=chat_model,
    prompt=mq_prompt,
    include_original=True,
)

class DedupRetriever(BaseRetriever):

    base_retriever: BaseRetriever

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:

        docs = self.base_retriever.get_relevant_documents(query)

        # deduplication
        seen: Set[str] = set()
        unique_docs: List[Document] = []

        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                unique_docs.append(d)

        return unique_docs
    
mq_retriever = DedupRetriever(base_retriever=mq_retriever_raw)


In [6]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        # measure response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

In [7]:
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
    timeout=600,      
    max_workers=2,    
    max_retries=2,    
    max_wait=600,     
)
    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config,
        )
    
    return result

In [8]:
metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),]

# Reranking: reranker models

Hybrid Retriever + Multi Query

In [ ]:
from FlagEmbedding import FlagReranker

local_path = "hf_models/rerank/bge-reranker-large"

reranker_model = FlagReranker(
    local_path,  
    use_fp16=True,
    devices='cuda:0'
)

In [ ]:
def rerank_docs(reranker_model, docs, query, top_k=10):
    pairs = [(query, d.page_content) for d in docs]

    # bge-reranker_model scoring
    scores = reranker_model.compute_score(pairs) 

    # reanking descent
    ranked = [
        doc for _, doc in sorted(
            zip(scores, docs),
            key=lambda x: x[0],
            reverse=True
        )
    ]
    return ranked[:top_k]

In [ ]:
class RerankRetriever(BaseRetriever):

    retriever: BaseRetriever
    reranker_model: object
    top_k: int = 10
    
    def _get_relevant_documents(self, query: str) -> List[Document]:

        docs = self.retriever.get_relevant_documents(query)
        
        reranked_docs = rerank_docs(self.reranker_model, docs, query, top_k=self.top_k)
        
        return reranked_docs


In [ ]:
k_values = [5, 10, 15]
mqre_rt_lst = []

In [ ]:
import time
import pandas as pd
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser

template = """You are a helpful assistant specializing in 3D printing operations.
Use the following pieces of retrieved context to answer the question.
If you don’t know the answer, just say that you don’t know.
Use two sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:
"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

for k in k_values:  
    mqre_retriever = RerankRetriever(retriever=mq_retriever,
                                    reranker_model=reranker_model,
                                    top_k=k,)
    mqre_rt_lst.append(mqre_retriever)

mqre_rag_chains = []
for retriever in mqre_rt_lst:
#  Build the RAG + reranker chain
    mqre_rag_chain = (
        {
            "context": retriever | RunnableLambda(format_docs),
            "question": RunnablePassthrough()
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    mqre_rag_chains.append(mqre_rag_chain)

In [ ]:
data_path = "evaluate_dataset/test_dataset.csv"

mqre_results_lst = []

for retriever, rag_chain in zip(mqre_rt_lst, mqre_rag_chains):
    eval_ds, response_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    mqre_results_lst.append((result,response_time))

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Evaluating: 100%|██████████| 200/200 [21:45<00:00,  6.53s/it]


In [ ]:
for idx, (result, response_time) in enumerate(mqre_results_lst):
    print(f'top-{k_values[idx]}hybrid_mq+reranker:', result)
    df = result.to_pandas()
    df["response_time"] = response_time
    df.to_csv(f"./evaluate_results/06_rerank_test/cross_encoder/top_{k_values[idx]}_bge.csv")

top-5hybrid+reranker: {'llm_context_precision_with_reference': 0.7076, 'context_recall': 0.5848, 'nv_accuracy': 0.4688, 'answer_relevancy': 0.9473, 'faithfulness': 0.7064}
top-10hybrid+reranker: {'llm_context_precision_with_reference': 0.6890, 'context_recall': 0.6751, 'nv_accuracy': 0.4437, 'answer_relevancy': 0.9460, 'faithfulness': 0.7780}
top-15hybrid+reranker: {'llm_context_precision_with_reference': 0.6913, 'context_recall': 0.7031, 'nv_accuracy': 0.4313, 'answer_relevancy': 0.9232, 'faithfulness': 0.8464}


# Reranking: Reranking language model scoring

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
tokenizer = AutoTokenizer.from_pretrained("./hf_models/rerank/qwen3-reranker-0.6b")
reranker_model = AutoModelForCausalLM.from_pretrained("./hf_models/rerank/qwen3-reranker-0.6b")

Using device: cuda


In [ ]:
import json
import torch
from typing import List, Any
from langchain.schema import Document
from langchain.schema.retriever import BaseRetriever
from langchain_core.runnables import RunnableLambda,RunnablePassthrough

class LLMRerankRetriever(BaseRetriever):
    base_retriever: BaseRetriever
    reranker_model: Any
    tokenizer: Any
    top_k: int = 10

    def _get_relevant_documents(self, query: str) -> List[Document]:
        docs = self.base_retriever.get_relevant_documents(query)
        if not docs:
            return []
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.reranker_model.to(device)
        scored = []
        for d in docs:
            prompt = f"Query: {query}\nPassage: {d.page_content}\nRelevant:"
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(device)
            with torch.no_grad():
                outputs = self.reranker_model(**inputs)
            logits = outputs.logits[0, -1]
            yes_id = self.tokenizer.convert_tokens_to_ids("yes")
            no_id = self.tokenizer.convert_tokens_to_ids("no")
            score = logits[yes_id] - logits[no_id]
            scored.append((d, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return [d for d, s in scored][:self.top_k]


MutiQuery + hybrid + qwen3 reranker model

In [ ]:
k_values = [5, 10, 15]
mqll_rt_lst = []

In [ ]:
for k in k_values:
    mqll_retriever = LLMRerankRetriever(
        base_retriever=mq_retriever,
        reranker_model=reranker_model,
        tokenizer=tokenizer,
        top_k=k
    )
    mqll_rt_lst.append(mqll_retriever)

In [ ]:
mqll_rag_chains = []
for retriever in mqll_rt_lst:
    rag_chain = (
        {"context": retriever | RunnableLambda(format_docs),
         "question": RunnablePassthrough()}
        | prompt
        | chat_model
        | StrOutputParser()
    )
    mqll_rag_chains.append(rag_chain)

In [ ]:
data_path = "evaluate_dataset/test_dataset.csv"

mqll_results_lst = []

for retriever, rag_chain in zip(mqll_rt_lst, mqll_rag_chains):
    eval_ds, response_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    mqll_results_lst.append((result,response_time))

Evaluating: 100%|██████████| 200/200 [22:16<00:00,  6.68s/it]


In [ ]:
for idx, (result, response_time) in enumerate(mqll_results_lst):
    print(f'top-{k_values[idx]}hybrid+llmscoring:', result)
    df = result.to_pandas()
    df["response_time"] = response_time
    df.to_csv(f"./evaluate_results/rerank_test/lm_model/top_{k_values[idx]}_qwen_mq.csv")

top-5hybrid+llmscoring: {'llm_context_precision_with_reference': 0.5932, 'context_recall': 0.5451, 'nv_accuracy': 0.4437, 'answer_relevancy': 0.9265, 'faithfulness': 0.6912}
top-10hybrid+llmscoring: {'llm_context_precision_with_reference': 0.5826, 'context_recall': 0.6251, 'nv_accuracy': 0.4250, 'answer_relevancy': 0.9492, 'faithfulness': 0.7420}
top-15hybrid+llmscoring: {'llm_context_precision_with_reference': 0.5488, 'context_recall': 0.6781, 'nv_accuracy': 0.4062, 'answer_relevancy': 0.9263, 'faithfulness': 0.8101}
